# RTCL-TP25K USB3 ユーザーモジュールサンプル

AXI4-Lite によるレジスタ読み書きや、AXI4-Stream によるデータ送受信を行うユーザーモジュールを、RTCL-TP25K-USB3 にて USB3.0 経由で PC から制御するサンプルです。

Verilog などの RTL開発に初めて触れる人が、まず usermodule.sv としてユーザーモジュールを書いてみることで PC から簡単にアクセスできる環境で FPGA の勉強を始められることを目的としています。

## rtcl_d3xx のインストール

Python から簡単に使えるように rtcl_d3xx を用意しています。インストールは以下のコマンドで行えます。

In [1]:
import struct
import time

# rtcl_d3xx のインポート
try:
    import rtcl_d3xx
except ImportError:
    # 未インストールの場合はリポジトリからインストールします。
    # Windows の場合は D3XX のインストール先を環境変数などに設定しておく必要があります。
    # rust/d3xx や python/rtcl-d3xx 以下の README を確認ください
    import subprocess
    import sys
    from pathlib import Path
    package_dir = Path.cwd() / "../../../../../python/rtcl-d3xx"
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", str(package_dir.resolve())]
    )
    import rtcl_d3xx

## FPGA 回路の合成

Python から FPGA 回路の合成を行ってみます。

gw_sh などの GOWIN のツールが動くように PATH などの環境変数を事前に設定しておいてください。

In [4]:
# FPGA 合成 (syn/cli/Makefile と同等のビルドを Windows/Linux 共通で実行)
import os
import shutil
import subprocess
import sys
from pathlib import Path

SYN_DIR = (Path.cwd() / "../../syn/cli").resolve()

# syn/cli からの相対パス (Makefile と同じ)
JELLY_DIR = "../../../../../jelly"
PRJ_DIR = "../.."
SHARED_DIR = "../../../../shared"
CONSTRAIN_DIR = "../../constrain"

TOP_MODULE = "rtcl_tp25k_usb3_usermodule_sample"
OUTPUT_BASE_NAME = TOP_MODULE
FS_FILE = SYN_DIR / "impl" / "pnr" / f"{OUTPUT_BASE_NAME}.fs"

VLOG_SOURCES = [
    f"{PRJ_DIR}/rtl/rtcl_tp25k_usb3_usermodule_sample.sv",
    f"{PRJ_DIR}/rtl/usermodule.sv",
    f"{SHARED_DIR}/rtl/ft601_multi_ch_mode.sv",
    f"{SHARED_DIR}/rtl/ft601_multi_ch_mode_transceiver.sv",
    f"{SHARED_DIR}/rtl/fifo32_cmd_axi4l.sv",
    f"{SHARED_DIR}/rtl/fifo32_cmd_axi4s_rx.sv",
    f"{SHARED_DIR}/rtl/fifo32_cmd_axi4s_tx.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_reset_async.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_stream_fifo.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_stream_fifo_async.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_fifo_async.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_fifo_fwft_read.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_ram_simple_dualport.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_data_async.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_capacity_async.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_capacity_buffer.sv",
    f"{JELLY_DIR}/rtl/v3/library/jelly3_stream_ff.sv",
    f"{JELLY_DIR}/rtl/v3/bus/jelly3_axi4l_if.sv",
    f"{JELLY_DIR}/rtl/v3/bus/jelly3_axi4s_if.sv",
    f"{JELLY_DIR}/rtl/v3/bus/jelly3_axi4s_packet_smoother.sv",
    f"{JELLY_DIR}/rtl/v3/bus/jelly3_axi4s_fifo.sv",
    f"{JELLY_DIR}/rtl/v3/primitive/jelly3_cdc_gray.sv",
    f"{JELLY_DIR}/rtl/v3/primitive/jelly3_cdc_array_single.sv",
    f"{JELLY_DIR}/rtl/v3/primitive/jelly3_cdc_single.sv",
    f"{PRJ_DIR}/ip/pll_init.v",
    f"{PRJ_DIR}/ip/gowin_pll/gowin_pll.v",
    f"{PRJ_DIR}/ip/gowin_pll/gowin_pll_mod.v",
    f"{PRJ_DIR}/ip/gowin_pll/gowin_pll_ft601.v",
    f"{PRJ_DIR}/ip/gowin_pll/gowin_pll_ft601_mod.v",
]
SDC_SOURCES = [f"{CONSTRAIN_DIR}/{TOP_MODULE}.sdc"]
CST_SOURCES = [f"{CONSTRAIN_DIR}/{TOP_MODULE}.cst"]

build_env = os.environ | {
    "DEVICE_PART_NUMBER": "GW5A-LV25MG121NC1/I0",
    "DEVICE_NAME": "GW5A-25A",
    "TOP_MODULE": TOP_MODULE,
    "OUTPUT_BASE_NAME": OUTPUT_BASE_NAME,
    "VLOG_SOURCES": " ".join(VLOG_SOURCES),
    "SDC_SOURCES": " ".join(SDC_SOURCES),
    "CST_SOURCES": " ".join(CST_SOURCES),
    "USE_CPU_AS_GPIO": "1",
    "USE_I2C_AS_GPIO": "1",
    "USE_DONE_AS_GPIO": "1",
    "USE_READY_AS_GPIO": "1",
    "PLACE_OPTION": "2",
    "ROUTE_OPTION": "2",
    "CO_PLACE_IO_REGISTERS": "1",
}


def run_streamed(cmd, check=True, cwd=None):
    # 出力をノートブックのセルに逐次表示しながら実行する
    proc = subprocess.Popen(
        cmd,
        cwd=cwd if cwd is not None else SYN_DIR,
        env=build_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {' '.join(cmd)}")


# make と同様にソースより .fs が新しければビルドをスキップ
sources = [SYN_DIR / s for s in (*VLOG_SOURCES, *SDC_SOURCES, *CST_SOURCES)]
missing = [s for s in sources if not s.exists()]
if missing:
    raise FileNotFoundError(f"ソースファイルが見つかりません: {missing}")

if FS_FILE.exists() and FS_FILE.stat().st_mtime >= max(s.stat().st_mtime for s in sources):
    print(f"up to date: {FS_FILE}")
else:
    if shutil.which("gw_sh") is None:
        raise RuntimeError(
            "gw_sh が見つかりません。Gowin EDA の IDE/bin ディレクトリを PATH に追加してください。"
        )
    run_streamed(["gw_sh", f"{JELLY_DIR}/scripts/gowin_build.tcl"])

# タイミングレポートのチェック (Makefile 同様にエラーは無視)
run_streamed(
    [
        sys.executable,
        f"{JELLY_DIR}/scripts/gowin_check_timing_report.py",
        f"impl/pnr/{TOP_MODULE}_tr_content.html",
    ],
    check=False,
)

up to date: D:\home\ryuji\git_work\rtcl-designs\projects\rtcl_tp25k_usb3\rtcl_tp25k_usb3_usermodule_sample\syn\cli\impl\pnr\rtcl_tp25k_usb3_usermodule_sample.fs


In [5]:
# FPGA のロード (syn/cli の make load と同等)
# LOADER を "programmer_cli" か "openFPGALoader" に書き換えることで使うツールを選べます。
LOADER = "programmer_cli"

if not FS_FILE.exists():
    raise FileNotFoundError(f".fs ファイルがありません。先に合成セルを実行してください: {FS_FILE}")

loader_path = shutil.which(LOADER)
if loader_path is None:
    raise RuntimeError(f"{LOADER} が見つかりません。PATH に追加してください。")

if LOADER == "programmer_cli":
    # GOWIN の programmer_cli で書き込み (ケーブル位置は programmer_cli --scan-cable で確認できます)
    programmer_cli_options = os.environ.get("PROGRAMMER_CLI_OPTIONS", "--location 11555").split()
    # programmer_cli はモジュールをカレントディレクトリ基準で探すため、インストール先を cwd にして実行する
    run_streamed(
        [loader_path, "--device", "GW5A-25B", "--run", "2", "--fsFile", str(FS_FILE)]
        + programmer_cli_options,
        cwd=Path(loader_path).parent,
    )
else:
    # openFPGALoader (Digilent HS2) で書き込み
    run_streamed(["openFPGALoader", "-c", "digilent_hs2", str(FS_FILE)])

 Target Cable: Gowin USB Cable(FT2CH)/0/11555/null@2.5MHz
 Target Device: GW5A-25B(0x0001281B)
 Operation "SRAM Program" for device#1...
 Frequency Updated: "15MHz"

Programing: [                         ] 1%                 
Programing: [                         ] 2%                 
Programing: [                         ] 3%                 
Programing: [#                        ] 4%                 
Programing: [#                        ] 5%                 
Programing: [#                        ] 6%                 
Programing: [#                        ] 7%                 
Programing: [##                       ] 8%                 
Programing: [##                       ] 9%                 
Programing: [##                       ] 10%                 
Programing: [##                       ] 11%                 
Programing: [###                      ] 12%                 
Programing: [###                      ] 13%                 
Programing: [###                      ] 14%       

In [ ]:
# デバイスを初期化
dev = rtcl_d3xx.Fifo32(dev_index=0)

In [ ]:
# レジスタ定義
ADDR_ID = 0x0000_0000
ADDR_VERSION = 0x0000_0004
ADDR_USER0 = 0x0000_0008
ADDR_USER1 = 0x0000_000C
ADDR_PUSH_SW = 0x0000_0010
ADDR_DIP_SW = 0x0000_0014
ADDR_LED = 0x0000_0018
ADDR_PMOD = 0x0000_001C


In [ ]:

# AXI4-Lite レジスタ読み出し
for name, address in (
    ("ID", ADDR_ID),
    ("VERSION", ADDR_VERSION),
    ("PUSH_SW", ADDR_PUSH_SW),
    ("DIP_SW", ADDR_DIP_SW),
    ("USER0", ADDR_USER0),
    ("USER1", ADDR_USER1),
):
    print(f"read  {name:<7}: 0x{dev.read_axi4l(address):08x}")

# User レジスタ読み書き
print("write USER0   : wdata = 0x12345678 wstrb=0b1111")
dev.write_axi4l(ADDR_USER0, 0x12345678, strb=0b1111)
print("write USER1   : wdata = 0xfedcba98 wstrb=0b1111")
dev.write_axi4l(ADDR_USER1, 0xFEDCBA98, strb=0b1111)
print(f"read  USER0   : 0x{dev.read_axi4l(ADDR_USER0):08x}")
print(f"read  USER1   : 0x{dev.read_axi4l(ADDR_USER1):08x}")

print("write USER0   : wdata = 0xaa55aa55 wstrb=0b1010")
dev.write_axi4l(ADDR_USER0, 0xAA55AA55, strb=0b1010)
print("write USER1   : wdata = 0xaa55aa55 wstrb=0b0101")
dev.write_axi4l(ADDR_USER1, 0xAA55AA55, strb=0b0101)
print(f"read  USER0   : 0x{dev.read_axi4l(ADDR_USER0):08x}")
print(f"read  USER1   : 0x{dev.read_axi4l(ADDR_USER1):08x}")

read  ID     : 0x1234abcd
read  VERSION: 0x00010000
read  PUSH_SW: 0x00000000
read  DIP_SW : 0x00000003
read  USER0  : 0xaa34aa78
read  USER1  : 0xfe55ba55
write USER0   : wdata = 0x12345678 wstrb=0b1111
write USER1   : wdata = 0xfedcba98 wstrb=0b1111
read  USER0   : 0x12345678
read  USER1   : 0xfedcba98
write USER0   : wdata = 0xaa55aa55 wstrb=0b1010
write USER1   : wdata = 0xaa55aa55 wstrb=0b0101
read  USER0   : 0xaa34aa78
read  USER1   : 0xfe55ba55


In [ ]:
# LED / PMOD 点滅
for _ in range(3):
    print("LED ON")
    dev.write_axi4l(ADDR_LED, 0x03)
    dev.write_axi4l(ADDR_PMOD, 0xFF)
    time.sleep(0.5)
    print("LED OFF")
    dev.write_axi4l(ADDR_LED, 0x00)
    dev.write_axi4l(ADDR_PMOD, 0x00)
    time.sleep(0.5)

LED ON
LED OFF
LED ON
LED OFF
LED ON
LED OFF


In [ ]:

# AXI4-Stream データ送受信
input_data = list(range(1, 11))
print(f"input data: {input_data}")
input_bytes = struct.pack("<10I", *input_data)
dev.send_axi4s(input_bytes, tuser=0)

packet = dev.recv_axi4s(timeout=1.0)
if len(packet.data) != len(input_bytes):
    raise RuntimeError(
        f"AXI4-Stream receive size mismatch: {len(packet.data)} != {len(input_bytes)}"
    )
result = list(struct.unpack("<10I", packet.data))
print(f"result: {result}")


input data: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
result: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [ ]:
# デバイスを削除してクローズ
del dev

Device closed
